<a href="https://colab.research.google.com/github/suder54ul/LLMP/blob/main/module03_handson2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 03 - Hands-on 2: Tool Use and Function Calling with LangChain
**Large Language Models in Production**
**Open – May 2026**

**Facilitator:** A/P TAN Wee Kek  
**Email:** distwk@nus.edu.sg

**Learning Objectives**
- Understand why LLMs need external tools (Design Pattern 3)
- Define custom tools using LangChain
- Use `bind_tools()` to enable function calling
- Build a simple manual tool-calling loop
- Appreciate production implications (safeguards, risks, governance)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from datetime import datetime
from zoneinfo import ZoneInfo

# Use the exact model from the course
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.0,
    num_ctx=8192,
)

print("LLaMA 3.1 8B Instruct is ready via Ollama!")

## Part 1: Why LLMs Need Tools

In [ ]:
query = "What is 12345 multiplied by 6789? Also tell me the current date and time in Singapore."

response_no_tool = llm.invoke([HumanMessage(content=query)])
print("=== LLM WITHOUT TOOLS (often wrong or hallucinated) ===\n")
print(response_no_tool.content)

## Part 2: Define Production-Ready Tools

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Safely evaluate a mathematical expression."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_singapore_time() -> str:
    """Return the current date and time in Singapore (UTC+8)."""
    now = datetime.now(ZoneInfo("Asia/Singapore"))
    return now.strftime("%Y-%m-%d %H:%M:%S Singapore Time")

@tool
def get_exchange_rate(base: str, target: str) -> str:
    """Mock exchange rate tool (SGD to other currencies)."""
    rates = {"USD": 0.74, "EUR": 0.68, "JPY": 110.5, "MYR": 3.45}
    rate = rates.get(target.upper(), "Unknown currency")
    return f"1 {base.upper()} = {rate} {target.upper()}"

tools = [calculator, get_current_singapore_time, get_exchange_rate]
print("Tools defined:")
for t in tools:
    print(f"- {t.name}: {t.description}")

## Part 3: Enable Function Calling with bind_tools()

In [ ]:
llm_with_tools = llm.bind_tools(tools)

query = "Calculate 12345 * 6789 and tell me the current Singapore time."
response = llm_with_tools.invoke([HumanMessage(content=query)])

print("=== RAW TOOL CALLS GENERATED BY LLM ===\n")
print(response.tool_calls if hasattr(response, "tool_calls") and response.tool_calls else "No tool calls")

## Part 4: Execute Tools & Return Result to LLM (Manual Tool Loop)

In [ ]:
def execute_tools(tool_calls):
    results = []
    for tc in tool_calls:
        selected_tool = next((t for t in tools if t.name == tc["name"]), None)
        if selected_tool:
            result = selected_tool.invoke(tc["args"])
            results.append(f"{tc['name']} result: {result}")
    return "\n".join(results)

# Simple chain: LLM → Tools → LLM
tool_calls = response.tool_calls if hasattr(response, "tool_calls") and response.tool_calls else []
tool_results = execute_tools(tool_calls)

# Feed results back to LLM
final_prompt = f"User query: {query}\n\nTool results:\n{tool_results}\n\nFinal answer:"
final_response = llm.invoke([HumanMessage(content=final_prompt)])

print("=== FINAL ANSWER AFTER TOOL USE ===\n")
print(final_response.content)

## Part 5: Your Turn – Build Your Own Tool (15–20 minutes)

Create **one new tool** relevant to your organisation (examples below) and add it to the `tools` list. Then test it with a new query.

Ideas:
1. Mock employee database lookup
2. PDPA compliance checker
3. Risk assessment scorer
4. Mock internal system status checker
5. Any idea that you fancy but it must require tool suuport

Copy the `@tool` pattern from above.

In [ ]:
# Define your Python function to be used as a tool and bind the new tool to the LLM



In [ ]:
# Test it with a query that requires both the calculator and the new tool.



## Part 6: Group Discussion & Production Implications (10 minutes)

1. How much more reliable was the answer after using tools?
2. What new **risks** appear once the LLM can call tools? (refer to lecture slides)
3. What **safeguards** must be added in production? (permissioning, validation, human approval, logging, rate limits, etc.)
4. How does this pattern fit into the overall Production LLM Reference Architecture?

**Key Takeaway (from lecture):**  
Tool use makes LLMs far more powerful, but it also makes them far more risky. Production systems require strong governance, validation, and monitoring.